# XGBoost Training

In [1]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
import os
import joblib
import json

## Loading Data

In [6]:
X_train=pd.read_csv("X_train.csv")
Y_train=pd.read_csv("Y_train.csv")
X_test=pd.read_csv("X_test.csv")
Y_test=pd.read_csv("Y_test.csv")

## Training

In [7]:
negative = (Y_train == 0).sum().item()
positive = (Y_train == 1).sum().item()

scale_pos_weight = negative / positive

print("\nNegative samples:", negative)
print("Positive samples:", positive)
print("Scale Pos Weight:", scale_pos_weight)


Negative samples: 198277
Positive samples: 99138
Scale Pos Weight: 2.0000100869495046


In [8]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,

    scale_pos_weight=scale_pos_weight,

    objective="binary:logistic",
    eval_metric="logloss",

    random_state=42,
    n_jobs=-1
)

In [9]:
print("\nTraining XGBoost...")

xgb_model.fit(
    X_train,
    Y_train
)

print("Training completed.")


Training XGBoost...
Training completed.


In [10]:
Y_pred = xgb_model.predict(X_test)

Y_prob = xgb_model.predict_proba(X_test)[:, 1]

## Model Evaluation

In [11]:
print("\nClassification Report:")
print(classification_report(
    Y_test,
    Y_pred,
    target_names=["Benign", "Non-Benign"]
))

print("Accuracy:", accuracy_score(Y_test, Y_pred))
print("Precision:", precision_score(Y_test, Y_pred))
print("Recall:", recall_score(Y_test, Y_pred))
print("F1 Score:", f1_score(Y_test, Y_pred))
print("ROC-AUC:", roc_auc_score(Y_test, Y_prob))
print("PR-AUC:", average_precision_score(Y_test, Y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(Y_test, Y_pred))


Classification Report:
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00     84976
  Non-Benign       0.79      0.80      0.80       142

    accuracy                           1.00     85118
   macro avg       0.90      0.90      0.90     85118
weighted avg       1.00      1.00      1.00     85118

Accuracy: 0.9993185930120538
Precision: 0.7916666666666666
Recall: 0.8028169014084507
F1 Score: 0.7972027972027972
ROC-AUC: 0.9673492731004745
PR-AUC: 0.8136916201529485

Confusion Matrix:
[[84946    30]
 [   28   114]]


## Saving Model

In [13]:
os.makedirs("models", exist_ok=True)

joblib.dump(
    xgb_model,
    "models/XGBoost_model.pkl"
)

['models/XGBoost_model.pkl']

## Saving Metrics

In [15]:
os.makedirs("XGBoost", exist_ok=True)
metrics = {
    "accuracy": accuracy_score(Y_test, Y_pred),
    "precision": precision_score(Y_test, Y_pred),
    "recall": recall_score(Y_test, Y_pred),
    "f1_score": f1_score(Y_test, Y_pred),
    "roc_auc": roc_auc_score(Y_test, Y_prob),
    "pr_auc": average_precision_score(Y_test, Y_prob)
}

with open("XGBoost/metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("Metrics saved.")

Metrics saved.


In [16]:
cm = confusion_matrix(Y_test, Y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual_Positive", "Actual_Negative"],
    columns=["Predicted_Positive", "Predicted_Negative"]
)

cm_df.to_csv(
    "XGBoost/confusion_matrix.csv"
)

print("Confusion matrix saved.")

Confusion matrix saved.


In [17]:
report = classification_report(
    Y_test,
    Y_pred,
    target_names=["Benign", "Non-Benign"]
)

with open("XGBoost/classification_report.txt", "w") as f:
    f.write(report)

print("Classification report saved.")

Classification report saved.


## Saving Predictions

In [20]:
predictions = pd.DataFrame({
    "actual": Y_test.values.ravel(),
    "prediction": Y_pred,
    "probability_non_benign": Y_prob
})

predictions.to_csv(
    "XGBoost/predictions.csv",
    index=False
)